## nb_inference

**Hourly inference** — reads the last 6 hours from `live_feed/rolling_buffer/` (actuals)
plus the next 8 hours from `live_feed/class_a_schedule/` (future schedule), builds the
6-snapshot temporal graph sequence, runs the champion `Seq2SeqGNN`, and writes 350 rows
(70 airports × 5 horizons) to two Delta tables:

- `predictions_latest` (overwritten each run) — DirectLake source for Power BI
- `predictions_history` (append-only) — audit + trend lines

**Output columns:** `airport_code`, `horizon_h`, `predicted_arr_delay_min`,
`pct_arr_delayed_15`, `target_ts`, `prediction_ts`, plus the enrichment columns
`sched_arr_count`, `sched_dep_count`, `expected_delayed_flights` (= pct × sched_arr)
that power the live dashboard's "expected delayed flights" and "departures in the next
N hours" cards. This is the same long schema the local export
(`scripts/export_powerbi.py` → `fact_predictions`) produces, so the cloud and offline
Power BI pages are interchangeable.

> **Note (schema/semantics change):** `pct_arr_delayed_15` now applies `sigmoid` to the
> classifier logit (channel 2) — previously the logit was clipped to [0, 1], which was
> incorrect. The enrichment columns are new. After importing this updated notebook,
> **re-run it once** (and let `predictions_history` `mergeSchema`) so both Delta tables pick
> up the corrected values and new columns. No wheel rebuild is needed — the notebook's
> own imports are unchanged.

**Trigger:** `pl_hourly_predict` fires at `:15` UTC each hour, 10 minutes after
`pl_fake_ingestion` writes the new buffer partition.

**One-time setup before first run:**
1. Build the wheel locally: `python -m build --wheel` → `dist/flight_delay_propagation-0.1.0-py3-none-any.whl`
2. Upload the wheel to OneLake → `Files/packages/`
3. Attach `FlightData_Lakehouse` as the default lakehouse for this notebook
4. (Recommended) Attach a Fabric custom environment `env_inference` with `torch>=2.1`, `torch-geometric>=2.4`, `deltalake>=0.14`, `pyarrow>=14` pre-installed — cuts cell 3 from ~5min cold-install down to ~10s.

In [ ]:
# Config — paths use the mounted lakehouse so no ADLS auth is needed.
HORIZONS = [1, 2, 4, 6, 8]
INPUT_WINDOW = 6  # snapshots fed to the seq2seq encoder
BUFFER_HOURS_READ = (
    24  # read more than INPUT_WINDOW so we tolerate gaps + low-traffic hours
)
MIN_ROUTE_FLIGHTS = (
    2  # training used 30 over 2 years; for inference, 2 is the equivalent noise filter
)
WINDOW_HOURS = 1
DEVICE = "cpu"  # F2 trial has no GPU; CPU inference is ~1–2s for 6 snapshots

LAKEHOUSE_FILES = "/lakehouse/default/Files"
MODEL_DIR = f"{LAKEHOUSE_FILES}/models/champion"
BUFFER_ROOT = f"{LAKEHOUSE_FILES}/live_feed/rolling_buffer"
SCHEDULE_ROOT = f"{LAKEHOUSE_FILES}/live_feed/class_a_schedule"
PACKAGES_ROOT = f"{LAKEHOUSE_FILES}/packages"

# Delta writes resolve table names against the attached default lakehouse via
# Spark — no explicit Tables paths needed.

In [ ]:
# Install the project wheel from the lakehouse. With env_inference attached,
# torch / torch-geometric / deltalake are already present and this finishes in ~10s.
import glob
import subprocess
import sys

wheels = sorted(glob.glob(f"{PACKAGES_ROOT}/flight_delay_propagation*.whl"))
if not wheels:
    raise FileNotFoundError(
        f"No wheel under {PACKAGES_ROOT}. Build locally with "
        f"'python -m build --wheel' and upload the .whl to Files/packages/."
    )

subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", wheels[-1]])
print(f"Installed: {wheels[-1].split('/')[-1]}")

In [ ]:
import json
from pathlib import Path

import pandas as pd
import torch

from src.data.graph_builder import (
    WeatherLookups,
    build_edge_index,
    create_temporal_graphs,
)
from src.models.factory import build_model
from src.utils.io import load_checkpoint_inference_only

In [ ]:
# Load the four champion artifacts uploaded in Phase 0d.
airport_map = json.loads(Path(f"{MODEL_DIR}/airport_map.json").read_text())
feature_stats = torch.load(
    f"{MODEL_DIR}/feature_stats.pt", map_location=DEVICE, weights_only=True
)
metadata = json.loads(Path(f"{MODEL_DIR}/metadata.json").read_text())

checkpoint_path = f"{MODEL_DIR}/{metadata['checkpoint_file']}"

print(f"Airports:   {len(airport_map)}")
print(f"Horizons:   {metadata.get('prediction_horizons', HORIZONS)}")
print(f"Checkpoint: {Path(checkpoint_path).name}")
print(f"Norm stats: mean shape={tuple(feature_stats['mean'].shape)}")

In [ ]:
# Read the last BUFFER_HOURS_READ partitions of rolling_buffer (actuals, CLASS_B_COLS).
# We read more than INPUT_WINDOW so that hours with <10 flights (graph_builder's
# minimum) don't cause us to drop below the seq2seq input length.
now_utc = pd.Timestamp.utcnow().floor("h")
frames_b, missing_b = [], []

for offset in range(BUFFER_HOURS_READ, 0, -1):
    ts = now_utc - pd.Timedelta(hours=offset)
    path = (
        Path(BUFFER_ROOT)
        / f"{ts.year}/{ts.month:02d}/{ts.day:02d}/{ts.hour:02d}/flights.parquet"
    )
    if path.exists():
        frames_b.append(pd.read_parquet(path))
    else:
        missing_b.append(ts)

if missing_b:
    print(f"WARNING: {len(missing_b)} buffer partitions missing")

if len(frames_b) < INPUT_WINDOW:
    raise RuntimeError(
        f"Only {len(frames_b)}/{INPUT_WINDOW} buffer partitions available. "
        f"Run nb_backfill_buffer first or wait for pl_fake_ingestion to catch up."
    )

df_buffer = pd.concat(frames_b, ignore_index=True)
print(
    f"Buffer: {len(df_buffer):,} rows across {len(frames_b)} partitions "
    f"({now_utc - pd.Timedelta(hours=BUFFER_HOURS_READ)} → {now_utc})"
)

In [ ]:
# Read the most recent class_a_schedule file. Each file contains 9 hours of
# forward schedule data INSIDE it (see GetFlightData function: it writes
# CRSDepTime in [H*100, (H+9)*100) under the path of the hour it ran).
# So a single recent file is enough — we don't iterate forward through hour
# paths because those paths don't exist (the function only writes at the
# timestamp it ran, not at each future hour the schedule covers).
df_schedule = pd.DataFrame()
loaded_from = None
for offset in range(0, 24):
    ts = now_utc - pd.Timedelta(hours=offset)
    path = (
        Path(SCHEDULE_ROOT)
        / f"{ts.year}/{ts.month:02d}/{ts.day:02d}/{ts.hour:02d}/schedule.parquet"
    )
    if path.exists():
        df_schedule = pd.read_parquet(path)
        loaded_from = ts
        break

if df_schedule.empty:
    print("WARNING: No class_a_schedule files found in the last 24h")
else:
    age_h = (now_utc - loaded_from).total_seconds() / 3600
    print(
        f"Schedule: {len(df_schedule):,} rows from {loaded_from} "
        f"(T-{age_h:.0f}h, contains 9h of forward schedule)"
    )

df_all = pd.concat([df_buffer, df_schedule], ignore_index=True)
print(f"Combined: {len(df_all):,} rows fed to the graph builder")

In [ ]:
# Build the airport graph (3 static edge features) and the snapshot list.
edge_index, edge_attr_static, edge_pairs = build_edge_index(
    df_all,
    airport_map,
    min_flights=MIN_ROUTE_FLIGHTS,
)
print(f"Edges: {edge_index.shape[1]} (min_flights={MIN_ROUTE_FLIGHTS})")

# Peek into the checkpoint to find the input_dim it was trained on. We can't
# trust feature_stats.shape because the uploaded champion artifacts came from
# possibly inconsistent runs (checkpoint with weather, stats without). The
# first GAT layer (spatial_encoder.input_proj.lin_l.weight) has shape
# [hidden*heads, input_dim], so its column count is what we need.
_ckpt_peek = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
_sd = _ckpt_peek["model_state_dict"]
_first_input_key = next(
    (k for k in _sd.keys() if "input_proj" in k and "lin_l.weight" in k),
    None,
)
if _first_input_key is None:
    raise RuntimeError("Could not auto-detect input_dim from checkpoint state_dict")
checkpoint_input_dim = _sd[_first_input_key].shape[1]
del _ckpt_peek, _sd
print(f"Checkpoint expects input_dim={checkpoint_input_dim}")

# 109 → trained with weather block (5 horizons: 9 hist + 9*5 future cols).
# 55  → trained without weather.
# Passing WeatherLookups() (empty dict) produces 109-dim features with the
# weather block zero-filled — same shape as training, zero signal for weather.
empty_weather = WeatherLookups() if checkpoint_input_dim == 109 else None
print(f"Weather block: {'enabled (zero-fill)' if empty_weather else 'disabled'}")

graphs = create_temporal_graphs(
    df=df_all,
    airport_map=airport_map,
    edge_index=edge_index,
    edge_attr_static=edge_attr_static,
    edge_pairs=edge_pairs,
    window_hours=WINDOW_HOURS,
    prediction_horizons=HORIZONS,
    weather_lookups=empty_weather,
)

input_dim = graphs[0].x.shape[1] if graphs else None
print(f"Snapshots built: {len(graphs)} (input_dim={input_dim})")

if len(graphs) < INPUT_WINDOW:
    raise RuntimeError(
        f"Only {len(graphs)} snapshots formed; need at least {INPUT_WINDOW}."
    )

In [ ]:
# Take the most recent INPUT_WINDOW snapshots and apply frozen normalization.
sequence = graphs[-INPUT_WINDOW:]
mean = feature_stats["mean"]
std = feature_stats["std"]

# If feature_stats was saved from a smaller-dim training run than what this
# checkpoint expects (known artifact-mismatch: uploaded champion had 55-dim
# stats but 109-dim checkpoint), pad with mean=0/std=1 for the trailing
# columns. The graph zero-fills those columns anyway, so passthrough is the
# semantically-correct behavior (weather treated as "no signal").
graph_dim = sequence[0].x.shape[1]
if mean.shape[0] < graph_dim:
    pad_size = graph_dim - mean.shape[0]
    mean = torch.cat([mean, torch.zeros(pad_size, dtype=mean.dtype)])
    std = torch.cat([std, torch.ones(pad_size, dtype=std.dtype)])
    print(
        f"WARNING: feature_stats ({feature_stats['mean'].shape[0]}) "
        f"< graph ({graph_dim}); padded with mean=0/std=1 for the "
        f"trailing {pad_size} columns"
    )
elif mean.shape[0] > graph_dim:
    raise RuntimeError(
        f"feature_stats has {mean.shape[0]} dims but graph only {graph_dim} — "
        f"can't safely truncate; check that cell 8 used the right weather setting."
    )

for g in sequence:
    g.x = (g.x - mean) / std

print(f"Sequence length: {len(sequence)}")
print(f"First snapshot ends at: {sequence[0].timestamp}")
print(f"Last snapshot ends at:  {sequence[-1].timestamp}")

In [ ]:
# Build the model with the same dims as configs/weekend/seq2seq_gnn_large.yaml.
# loss="multi_task" forces the factory to wire output_channels=3 (arr_delay,
# dep_delay, pct_delayed); we only consume channels 0 and 2.
config = {
    "model": {
        "name": "seq2seq_gnn",
        "seq2seq_gnn": {
            "hidden_dim": 256,
            "num_heads": 8,
            "num_spatial_layers": 3,
            "num_temporal_layers": 2,
            "dropout": 0.3,
        },
    },
    "graph": {"prediction_horizons": HORIZONS},
    "training": {"loss": "multi_task"},
}

model = build_model(config, input_dim=input_dim, edge_dim=5).to(DEVICE)
ckpt = load_checkpoint_inference_only(checkpoint_path, model, device=DEVICE)
model.eval()

print(f"Loaded checkpoint epoch={ckpt['epoch']}")
print(f"Training metrics:   {ckpt.get('metrics', {})}")

In [ ]:
# Forward pass → [N, H, C] = [70, 5, 3]. Channel 0 = ArrDelay (min),
# channel 2 = logit of P(delay≥15).
with torch.no_grad():
    preds = model(sequence)

arr_delay = preds[:, :, 0].cpu().numpy()
# Channel 2 is a LOGIT (the BCEWithLogits head keeps the sigmoid OUTSIDE the
# model — see seq2seq_gnn head comment). Apply sigmoid to get a probability in
# [0, 1]. Previously this clipped the raw logit to [0, 1], which silently
# mis-reported the delay probability — fixed here to match metrics.py and
# src/inference/predictor.py.
pct_15 = torch.sigmoid(preds[:, :, 2]).cpu().numpy()

idx_to_iata = {v: k for k, v in airport_map.items()}

rows = []
for node_idx in range(len(airport_map)):
    iata = idx_to_iata[node_idx]
    for h_idx, h in enumerate(HORIZONS):
        target_ts = now_utc + pd.Timedelta(hours=h)
        rows.append(
            {
                "airport_code": iata,
                "horizon_h": int(h),
                "predicted_arr_delay_min": float(arr_delay[node_idx, h_idx]),
                "pct_arr_delayed_15": float(pct_15[node_idx, h_idx]),
                "target_ts": target_ts.isoformat(),
                "prediction_ts": now_utc.isoformat(),
            }
        )

df_preds = pd.DataFrame(rows)
print(
    f"Predictions: {len(df_preds)} rows "
    f"({df_preds['airport_code'].nunique()} airports × "
    f"{df_preds['horizon_h'].nunique()} horizons)"
)
df_preds.head(10)

In [ ]:
# --- Enrichment: scheduled volume per (airport, target hour) -----------------
# Adds sched_arr_count / sched_dep_count from the forward schedule (df_schedule)
# so the live Power BI dashboard can compute "expected delayed flights"
# (pct_arr_delayed_15 x sched_arr_count) and "flights departing in the next N
# hours" (sched_dep_count). These mirror the columns that the local export
# (scripts/export_powerbi.py -> fact_predictions) attaches offline.


def _hourly_counts(df, code_col, time_col, rollover_against=None):
    """Map {(airport_code, naive-UTC hour): scheduled count} from HHMM times."""
    if df is None or df.empty:
        return {}
    needed = [code_col, "FlightDate", time_col]
    if rollover_against:
        needed.append(rollover_against)
    d = df[needed].dropna(subset=[code_col, "FlightDate", time_col]).copy()
    hhmm = pd.to_numeric(d[time_col], errors="coerce")
    hour = (hhmm // 100).clip(0, 23).fillna(0).astype(int)
    date = pd.to_datetime(d["FlightDate"]).dt.normalize()
    if rollover_against:  # arrival scheduled before departure -> next calendar day
        ref = pd.to_numeric(d[rollover_against], errors="coerce")
        overnight = (hhmm < ref).fillna(False).astype(int)
        date = date + pd.to_timedelta(overnight, unit="D")
    ts = date + pd.to_timedelta(hour, unit="h")
    out = pd.DataFrame({"c": d[code_col].astype(str).values, "t": ts.values})
    return out.groupby(["c", "t"]).size().to_dict()


arr_counts = _hourly_counts(
    df_schedule, "Dest", "CRSArrTime", rollover_against="CRSDepTime"
)
dep_counts = _hourly_counts(df_schedule, "Origin", "CRSDepTime")

# target_ts may be tz-aware (+00:00); normalize to naive UTC to match the keys.
_tgt = (
    pd.to_datetime(df_preds["target_ts"], utc=True).dt.tz_localize(None).dt.floor("h")
)
_codes = df_preds["airport_code"].astype(str)
df_preds["sched_arr_count"] = [
    int(arr_counts.get((c, t), 0)) for c, t in zip(_codes, _tgt)
]
df_preds["sched_dep_count"] = [
    int(dep_counts.get((c, t), 0)) for c, t in zip(_codes, _tgt)
]
df_preds["expected_delayed_flights"] = (
    df_preds["pct_arr_delayed_15"] * df_preds["sched_arr_count"]
).round(2)

print(
    f"sched_arr_count total={int(df_preds['sched_arr_count'].sum())}, "
    f"sched_dep_count total={int(df_preds['sched_dep_count'].sum())}"
)
df_preds[
    [
        "airport_code",
        "horizon_h",
        "pct_arr_delayed_15",
        "sched_arr_count",
        "sched_dep_count",
        "expected_delayed_flights",
    ]
].head(10)

In [ ]:
# Write predictions via Spark — the deltalake Rust client trips a TLS chain
# error (CaUsedAsEndEntity) on Fabric's egress, and there's no Python-side
# knob to relax it. Spark's ABFSS driver uses the workspace identity natively
# and resolves saveAsTable against the attached default lakehouse.
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

sdf = spark.createDataFrame(df_preds)

# overwriteSchema/mergeSchema let the tables absorb the enrichment columns
# (sched_arr_count, sched_dep_count, expected_delayed_flights) the first time
# this enriched notebook runs over a pre-existing table.
(
    sdf.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("predictions_latest")
)

(
    sdf.write.format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable("predictions_history")
)

print(f"predictions_latest:  {sdf.count()} rows (overwrite)")
print(f"predictions_history: {sdf.count()} rows appended")

In [ ]:
# Verification — confirms exactly 350 rows, 70 distinct airports, plausible delay range.
df_check = spark.read.table("predictions_latest").toPandas()

print(f"predictions_latest: {len(df_check)} rows")
print(f"  airports: {df_check['airport_code'].nunique()}")
print(f"  horizons: {sorted(df_check['horizon_h'].unique().tolist())}")
print(f"  prediction_ts: {df_check['prediction_ts'].iloc[0]}")
print()
print("Delay distribution by horizon (minutes):")
print(
    df_check.groupby("horizon_h")["predicted_arr_delay_min"]
    .agg(["mean", "min", "max"])
    .round(2)
)